# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import gradio as gr




So let's look at what this week one exercise is.

I want to see your familiarity with OpenAI and Olama to build a tool that takes a technical question,

which you can just type right here and responds with an explanation of that question.

So I've got a little couple of hints down below, but use this as a way to call a model so that you

could retype a question in here, and the model will explain it.

In [ ]:
# constants

MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_LLAMA = 'meta-llama/llama-guard-4-12b'
MODEL_ANTHROPIC = "anthropic/claude-opus-4.6"
OLLAMA_BASE_URL = "http://localhost:11434/v1"



In [ ]:
# set up environment
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY') 

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    




In [ ]:


openrouter = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv('OPENROUTER_API_KEY'), 
)

openai = OpenAI()


In [ ]:


ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
# here is the question; type over this to ask something new

question = """
You are a technical assistant, and you are extremely good in python. 
You will take a technical question and respond with an explanation of that question. 
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
And also answer other python questions asked.
"""
messages = [{"role": "user", "content": question}]




#  Streaming `without` using display and Markdown class from IPython.display 

In [ ]:
# Get gpt-4o-mini to answer, with streaming
# Streaming with out using display and Markdown class from IPython.display 
def call_gpt_4o():
    response = []
    stream = openrouter.chat.completions.create( model= MODEL_GPT,messages= messages , stream=True) 
    for chunk in stream:
        try:
            delta = chunk.choices[0].delta
            if delta.content:
                print(delta.content, end="", flush=True)
                response.append(delta.content) 
        except (IndexError, AttributeError): 
            continue
    return response 
    

print(call_gpt_4o())

#  Streaming `WITH` using display and Markdown class from IPython.display 

In [ ]:
def call_gpt_4o():
    response = ''
    stream = openrouter.chat.completions.create( model= MODEL_GPT,messages= messages , stream=True) 
    display_handle = display(Markdown(""), display_id=True)

    

    for chunk in stream:
        try:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id) 
        except (IndexError, AttributeError): 
            print("Error: IndexError or AttributeError")
            continue    
    return response 
    

print(call_gpt_4o())

In [ ]:
def call_gpt_4o():
    response = ''
    stream = openrouter.chat.completions.create( model= MODEL_GPT,messages= messages , stream=True) 
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        try:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id) 
        except (IndexError, AttributeError): 
            print("Error: IndexError or AttributeError")
            continue    
    

print(call_gpt_4o())

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
def call_llama_3_2():
    #   
    try:
        response = ''
        stream = ollama.chat.completions.create(model= "llama3.2",messages= messages , stream=True)
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id) 
        return response 
    except Exception as e:
        print(f"Error: {e}") 
        return None   
    
    
print(call_llama_3_2()) 

In [ ]:
# Get Llama 3.2 to answer

def call_llama_3_3():
    #   
    try:
        response = ''
        stream = openrouter.chat.completions.create(model= MODEL_LLAMA,messages= messages , stream=True)
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            if chunk.choices and len(chunk.choices) > 0:
                delta = chunk.choices[0].delta
                content = getattr(delta, 'content', None)
                if content:
                    response += content
                    
        return response
    except Exception as e:
        print(f"Error: {e}") 
    
    
print(call_llama_3_3())

In [ ]:
def call_gpt_4o(prompt):
    messages = [{"role": "system", "content": question}, {"role": "user", "content": prompt}]
    response = ''
    stream = openrouter.chat.completions.create( model= MODEL_GPT,messages= messages , stream=True) 
    display_handle = display(Markdown(""), display_id=True)

    

    for chunk in stream:
            if chunk.choices and len(chunk.choices) > 0: 
                delta = chunk.choices[0].delta 
                content = getattr (delta, "content", None)
                if content:
                    response += content
                    yield response
    
# ***************************************************************************************************************
def call_llama_4_0(prompt):

    #   
    response = ''
    messages = [{"role": "system", "content": question}, {"role": "user", "content": prompt}]
    stream = openrouter.chat.completions.create(model= MODEL_LLAMA,messages= messages , stream=True)
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices and len(chunk.choices) > 0:
            delta = chunk.choices[0].delta
            content = getattr(delta, 'content', None)
            if content:
                response += content 
                yield response 


# ***************************************************************************************************************        

def call_opus_4_6(prompt):
    #   
    response = ''
    messages = [{"role": "system", "content": question}, {"role": "user", "content": prompt}]
    stream = openrouter.chat.completions.create(model= MODEL_ANTHROPIC,messages= messages , stream=True)
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices and len(chunk.choices) > 0:
            delta = chunk.choices[0].delta
            content = getattr(delta, 'content', None)
            if content:
                response += content 
                yield response  
# ****************************************************************************************************
def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

In [ ]:
def stream_model(prompt, model):
    if model=="GPT":
        result = call_gpt_4o(prompt)
    elif model=="Llama_4.0":
        result = call_llama_4_0(prompt)
    elif model=="Opus_4.6":
        result = call_opus_4_6(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Llama_4.0", "Opus_4.6"], 
label="Select model", value=["GPT", "Llama_4.0", "Opus_4.6"]
  )
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output],
     examples=[
        ["Please explain what this code does and why: yield from {book.get('author') for book in books if book.get('author')}", "GPT"]
    ], 
    flagging_mode="never"
    )
view.launch()